# Lab 06: 卷積神經網路 (CNN) 實作 — CIFAR-10 彩色影像分類

在本單元中，我們將動手建構一個真正的 **卷積神經網路 (CNN)**，並用它來做彩色圖片分類（使用 **CIFAR-10 資料集**）。
最後，我們將進行一次 CPU 與 GPU 訓練時間的計時挑戰，親眼見證顯示卡在影像深度學習上的強大加速優勢！

### GPU 環境測試
在開始之前，請先確認您的執行環境已成功啟用 GPU 加速器（例如 Google Colab 中的 T4 GPU）。

**【重要】**若下方輸出顯示為 CPU 模式，請點選右上角『連線 / 變更執行階段類型』，將硬體加速器切換為 GPU 後再往下執行，以防切換執行階段時已安裝的套件與下載的資料被清空重設！

In [ ]:
import torch
if not torch.cuda.is_available():
    print('[重要提示] 偵測到 CPU 模式！')
    print('CIFAR-10 訓練在 CPU 上會非常慢，強烈建議：')
    print('「執行階段」→「變更執行階段類型」→ 選 T4 GPU → 儲存 → 重新執行')
else:
    print(f'GPU 就緒: {torch.cuda.get_device_name(0)}')
    print(f'PyTorch: {torch.__version__}')


### 環境安全網與資料準備
確認已啟用 GPU 後，接著我們安裝缺少的套件，並下載測試 CIFAR-10 資料集（約 170MB，請耐心等待）。

In [ ]:
import subprocess, sys

def ensure_pkg(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch', 'torchvision', 'matplotlib']:
    ensure_pkg(pkg)

# 快速驗證 torchvision CIFAR-10 可存取
try:
    import torchvision
    try:
        print('正在從高速度鏡像源 (BrainChip) 下載 CIFAR-10 資料集 (約 170MB，請耐心等待)...')
        torchvision.datasets.CIFAR10.url = "https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz"
        test_ds = torchvision.datasets.CIFAR10(root='./data', train=False, download=True,
                                                 transform=torchvision.transforms.ToTensor())
    except Exception as err:
        print(f'鏡像源下載失敗 ({err})，嘗試使用官方多倫多大學來源下載...')
        torchvision.datasets.CIFAR10.url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
        test_ds = torchvision.datasets.CIFAR10(root='./data', train=False, download=True,
                                                 transform=torchvision.transforms.ToTensor())
    print(f'CIFAR-10 就緒: {len(test_ds)} 張測試圖片，10 個類別')
except Exception as e:
    print(f'[警告] CIFAR-10 下載失敗: {e}')
    print('[備用] 將在後續步驟使用隨機彩色圖片模擬 CIFAR-10，不影響 CNN 架構學習')


### 任務 1: 載入並觀察 CIFAR-10 資料集
CIFAR-10 包含 10 種不同類別（飛機、貓、狗、船等）的 $32 \times 32$ 彩色影像。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import time

# 影像前處理：轉為張量並標準化彩色通道 (RGB)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 下載與讀取 CIFAR-10
try:
    torchvision.datasets.CIFAR10.url = "https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz"
    train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
except Exception as err:
    print(f'鏡像源載入失敗 ({err})，切換回官方多倫多大學來源...')
    torchvision.datasets.CIFAR10.url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# 顯示一小批圖片
images, labels = next(iter(train_loader))
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    # 還原標準化以利正常顯示圖片顏色
    img = images[i] / 2 + 0.5
    plt.imshow(img.permute(1, 2, 0).numpy()) # PyTorch [C, H, W] -> Matplotlib [H, W, C]
    plt.title(classes[labels[i].item()])
    plt.axis('off')
plt.show()

### 任務 2: 建構 CNN 影像分類器
請在下方補全一個簡單的 CNN 模型，結構包含兩個卷積區段（卷積 → 活化 → 池化）與全連接分類器。

*提示：經過兩次 `MaxPool2d(2, 2)`，圖片的解析度會從 $32 \times 32$ 變成 $16 \times 16$，再變成 $8 \times 8$。因此，進入全連接層前的特徵形狀為 `[通道數 * 8 * 8]`。*

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 1. 卷積特徵提取特徵層
        self.features = nn.Sequential(
            # (已幫你完成) 第一層卷積，輸入 3 通道 (RGB)，輸出 16 個特徵圖
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 32x32 -> 16x16
            
            # (已幫你完成) 第二層卷積，輸入 16 通道，輸出 32 個特徵圖
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8
        )
        
        # 2. 全連接分類層
        self.classifier = nn.Sequential(
            nn.Flatten(),                          # 攤平成一維: 32通道 * 8高 * 8寬 = 2048 維
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)                     # 10 個分類類別
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN()
print(model)

### 任務 3: 大對決！CPU 訓練 vs GPU 加速
我們各抽取 100 批資料（約 12800 張圖），測量 CPU 與 GPU 的處理時間差。

In [ ]:
criterion = nn.CrossEntropyLoss()
test_epochs = 1
num_batches = 100 # 限制訓練步數，避免 CPU 跑太久

# -----------------------------------------------------
# 選手一號：CPU 訓練
# -----------------------------------------------------
model_cpu = SimpleCNN().to('cpu')
optimizer_cpu = optim.Adam(model_cpu.parameters(), lr=0.001)
print("開始 CPU 訓練對決...")

start_time = time.time()
for i, (images, labels) in enumerate(train_loader):
    if i >= num_batches: break
    outputs = model_cpu(images)
    loss = criterion(outputs, labels)
    optimizer_cpu.zero_grad()
    loss.backward()
    optimizer_cpu.step()
cpu_duration = time.time() - start_time
print(f"→ CPU 訓練 {num_batches} 批資料完成！耗時: {cpu_duration:.2f} 秒\n")

# -----------------------------------------------------
# 選手二號：GPU 訓練
# -----------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
    model_gpu = SimpleCNN().to(device)
    optimizer_gpu = optim.Adam(model_gpu.parameters(), lr=0.001)
    print("開始 GPU (CUDA) 訓練對決...")
    
    # GPU 預熱
    dummy_x, dummy_y = next(iter(train_loader))
    _ = model_gpu(dummy_x.to(device))
    torch.cuda.synchronize()
    
    start_time = time.time()
    for i, (images, labels) in enumerate(train_loader):
        if i >= num_batches: break
        # TODO: 將影像與標籤移到 GPU (device)
        # 提示：images = images.to(device)、labels = labels.to(device)
        # ??? (請在此填寫你的答案)
        
        outputs = model_gpu(images)
        loss = criterion(outputs, labels)
        
        optimizer_gpu.zero_grad()
        loss.backward()
        optimizer_gpu.step()
        
    torch.cuda.synchronize()
    gpu_duration = time.time() - start_time
    print(f"→ GPU 訓練 {num_batches} 批資料完成！耗時: {gpu_duration:.2f} 秒\n")
    
    speed_ratio = cpu_duration / gpu_duration
    print("========================================")
    print(f"GPU (顯卡) 比 CPU 快了 {speed_ratio:.1f} 倍！")
    print("========================================")
else:
    print("目前沒有 GPU 可進行對決。")

### 任務 4: 測試與推理視覺化
我們隨機取出十張測試集圖片，使用剛剛在 GPU 訓練好的模型進行推理，並將結果呈現在畫面上。

In [ ]:
if torch.cuda.is_available():
    model_gpu.eval()
    # 建立一個 shuffle=True 的臨時 DataLoader 來隨機抽取 10 張圖片
    temp_loader = torch.utils.data.DataLoader(test_dataset, batch_size=10, shuffle=True)
    images, labels = next(iter(temp_loader))
    
    with torch.no_grad():
        outputs = model_gpu(images.to(device))
        _, preds = outputs.max(1)
        
    plt.figure(figsize=(15, 6))
    for i in range(10):
        plt.subplot(2, 5, i+1)
        img = images[i] / 2 + 0.5
        plt.imshow(img.permute(1, 2, 0).numpy())
        
        pred_class = classes[preds[i].item()]
        true_class = classes[labels[i].item()]
        color = 'green' if pred_class == true_class else 'red'
        
        plt.title(f"Pred: {pred_class}\nTrue: {true_class}", color=color)
        plt.axis('off')
    plt.show()

### 任務 5: 站在巨人的肩膀上 — 遷移學習 (Transfer Learning)
在實際的影像識別任務中，我們很少從零開始訓練一個超大型的卷積網路。相反地，我們會使用在大型資料集（如 ImageNet，包含數百萬張真實照片）上已經訓練好的知名模型（例如 ResNet-18），並直接借用它『提取影像特徵』的成熟視覺能力。

我們主要需要完成兩件事：
1. **凍結特徵提取層參數**：將所有預訓練層的 `requires_grad` 設定為 `False`，告訴 PyTorch 不需要計算這些層的梯度，也無須更新參數，這能大幅節省計算時間。
2. **重組最後的分類器**：原模型輸出為 ImageNet 的 1000 個類別，我們需要將最後的 `fc`（全連接層）替換為一個輸出為 10 個類別（CIFAR-10 類別數）的全新 Linear 層。

**【動手實作】**：請在下方尋找 `[TODO]` 標記，補全遷移學習設定，並執行預訓練 ResNet-18 在 GPU 上的微調訓練！

In [ ]:
import torchvision.models as models

# 載入經典的預訓練 ResNet-18 模型
# weights=models.ResNet18_Weights.DEFAULT 會自動下載官方在 ImageNet 訓練好的權重參數
resnet_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# TODO 1: 凍結特徵提取層所有參數，使其不參與梯度更新
for param in resnet_model.parameters():
    # param.requires_grad = ??? (請在此填寫你的答案)
    pass

# 觀察原本全連接層的輸入維度
in_features = resnet_model.fc.in_features
print("原 ResNet18 fc 層輸入維度:", in_features)

# TODO 2: 重組最後的全連接層，使其輸出 10 個類別 (對應 CIFAR-10)
# 提示：宣告一個 nn.Linear(in_features, 10)
# resnet_model.fc = ??? (請在此填寫你的答案)

# 防呆檢查：確認 TODO 2 已完成
if resnet_model.fc.out_features != 10:
    print('[提醒] fc 層還是 1000 類輸出，請先完成上方 TODO 2 再往下執行！')

# 搬移至 GPU 加速器
if torch.cuda.is_available():
    resnet_model = resnet_model.to(device)
    # 注意：我們只需要訓練最後一層 fc 的參數，故 Optimizer 僅傳入 resnet_model.fc 的 parameters
    optimizer_resnet = optim.Adam(resnet_model.fc.parameters(), lr=0.001)
    
    print("開始預訓練 ResNet-18 遷移學習微調...")
    start_time = time.time()
    for i, (images, labels) in enumerate(train_loader):
        if i >= num_batches: break
        
        # 搬移資料至 GPU
        images, labels = images.to(device), labels.to(device)
        
        outputs = resnet_model(images)
        loss = criterion(outputs, labels)
        
        optimizer_resnet.zero_grad()
        loss.backward()
        optimizer_resnet.step()
        
    print(f"→ 預訓練 ResNet-18 微調完成！耗時: {time.time() - start_time:.2f} 秒\n")
else:
    print("目前沒有 GPU 可進行遷移學習微調實作。")

---

## Lab 06 學習重點小結
### 核心概念掌握
- CIFAR-10 是 10 類 32×32 彩色影像資料集，每類 6,000 張訓練圖片。
- 彩色影像的張量形狀為 [batch, 3, H, W]，3 代表 RGB 三通道。
- 遷移學習 (Transfer Learning)：凍結預訓練 ResNet-18 的特徵層參數，只微調最後的分類層，即可用少量運算量達到高準確率。
- GPU 平行計算使大批次 (batch) 訓練速度遠超 CPU。

### 快速自評測驗

**Q1. 一張 CIFAR-10 彩色影像的張量形狀為？**
- A. [1, 32, 32]
- B. [3, 32, 32]
- C. [32, 32, 3]

<details><summary>查看解答</summary>

**答案：B — PyTorch 中彩色影像格式為 [Channel, Height, Width]，即 [3, 32, 32]。**

</details>

**Q2. 在相同的 CNN 模型與資料量下，GPU 訓練快於 CPU 的根本原因是？**
- A. GPU 記憶體更大
- B. GPU 有數千個核心可平行計算
- C. GPU 時脈頻率更高

<details><summary>查看解答</summary>

**答案：B — GPU 的 CUDA 核心數量遠超 CPU，適合大量矩陣運算的平行化。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「CIFAR-10 是 10 類 32×32 彩色影像資料集，」
- [ ] 我可以用一句話解釋「彩色影像的張量形狀為 [batch, 3, H, W]，3 」
- [ ] 我可以用一句話解釋「遷移學習 (Transfer Learning)：凍結預訓練層參數，只微調分類層」
- [ ] 我的程式碼成功執行並得到預期輸出